# 3b - The risk table, built by hand

**Time:** 35 minutes  |  **AI: off** - you write every line yourself

**What you will do**
- Compute four measures for six assets: annual return, volatility, beta against the S&P 500, Sharpe ratio.
- Put them in one table, sort it, round it, send it to Excel.
- Check the table with a known case.

How to work: click a cell, press Shift+Enter to run it. Run the cells in order, top to bottom. Fill each blank, marked `YOUR CODE HERE`. Use the pandas you met in 3a.

## The problem

The MD wants one table for the five mega-caps and the S&P 500: one row per asset, one column per measure.

| | Annual return | Volatility | Beta vs S&P 500 | Sharpe (rf 3%) |
|---|---|---|---|---|
| Apple | ... | ... | ... | ... |
| ... | | | | |

- **Annual return**: the compound yearly growth of the price over the five years.
- **Volatility**: as in 3a, the annualized standard deviation of the daily returns.
- **Beta**: how much the stock moves when the S&P 500 moves. Beta 1.2 means: when the index moves 1%, the stock tends to move 1.2%. It is the covariance (how much two series move together) of the stock with the S&P 500, divided by the variance of the S&P 500.
- **Sharpe ratio**: return above the risk-free rate, per unit of risk: `(annual return - rf) / volatility`. The risk-free rate `rf` is what you earn without any risk, for example from a government bond. Here it is 3%.

In [ ]:
import pandas as pd

# Given: the prices and the daily returns, as in 3a
DATA = "https://raw.githubusercontent.com/niparis/gem-python-finance/main/data/"
prices = pd.read_excel(DATA + "peers.xlsx", index_col="Date")
returns = prices.pct_change().dropna()

days = len(returns)    # number of daily returns
rf = 0.03              # risk-free rate, a decimal (3%)

print(days)

## Step 1 - Annual return

Compound growth: divide the last price by the first price, raise the result to the power `252 / days` (`**` is the power), then subtract 1. Mind the brackets: `(a / b) ** (c / d) - 1`. Do not use the average of the daily returns: that would ignore compounding.

`prices.iloc[-1]` is the last row, `prices.iloc[0]` the first row. Pandas divides column by column: the result has one number per asset.

You should get Apple **19.6%** and the S&P 500 **10.4%**.

In [ ]:
# YOUR CODE HERE: save the annual return of every asset in ann_return

print(ann_return.round(3))

## Step 2 - Volatility

The same line as in 3a, cell 5 (open 3a again if you need it).

You should get Apple **28.1%** and Meta **45.3%**.

In [ ]:
# YOUR CODE HERE: save the annualized volatility of every asset in vol

print(vol.round(3))

## Step 3 - Beta against the S&P 500

`returns.cov()` is like `returns.corr()`, but it gives covariances: like a correlation, but not squeezed between -1 and 1. The column `["SP500"]` of that table holds the covariance of every asset with the S&P 500. `returns["SP500"].var()` is the variance of the S&P 500 (the standard deviation squared).

Beta = covariance with the S&P 500 / variance of the S&P 500.

You should get Apple **1.16**, Meta **1.61**.

In [ ]:
# YOUR CODE HERE: save the beta of every asset in beta

print(beta.round(2))

## Step 4 - Sharpe ratio

`(ann_return - rf) / vol`: the return above the risk-free rate, per unit of volatility. `ann_return` and `vol` are Series (one column of numbers, one per asset): pandas works asset by asset.

You should get Apple **0.59**, Amazon **0.19**.

In [ ]:
# YOUR CODE HERE: save the Sharpe ratio of every asset in sharpe

print(sharpe.round(2))

## Step 5 - Put it in one table

`pd.DataFrame({...})` builds a table from a dictionary (Session 1): each **key** is a column name, each **value** is a Series with the same row labels. Then `.sort_values("Sharpe", ascending=False)` puts the biggest Sharpe first (`ascending=False` means biggest first), and `.round(3)` keeps 3 decimals: 0.196 reads as 19.6%. `.round(3)` gives a new table: `table` itself keeps every decimal.

In [ ]:
table = pd.DataFrame({
    "Annual return": ...,    # YOUR CODE HERE (replace each ... with the right Series)
    "Volatility": ...,
    "Beta": ...,
    "Sharpe": ...,
})

# YOUR CODE HERE: sort the table by Sharpe, best first, and round to 3 decimals
# save it in risk_table, then show it

risk_table

You should get this order: **Apple, S&P 500, Alphabet, Microsoft, Meta, Amazon**, with Sharpe 0.59, 0.43, 0.39, 0.27, 0.25, 0.19.

**Your answer:** which stocks beat the S&P 500 on Sharpe (return per unit of risk)? Meta earned more than the index: why is its Sharpe lower? Compare the Volatility column. Write 2 or 3 lines here: ...

## Check - The known case

**Known case** (checks card, entry 5): the beta of the S&P 500 against itself is exactly 1. Every correlation of an asset with itself is 1. If your code does not give this, it is wrong.

The guard stops the code with a message if the known case fails. It rounds to 6 decimals first, because computers keep tiny errors in decimals.

In [ ]:
sp_beta = beta["SP500"]

if round(sp_beta, 6) != 1:
    raise ValueError(f"Beta of the S&P 500 against itself is {sp_beta}, it must be 1.")
print("Known case OK: beta of the S&P 500 is 1.00")

# Same idea for the correlation of every asset with itself. .values.diagonal() takes the diagonal of the table: each asset against itself
self_corr = returns.corr().values.diagonal()
if (self_corr.round(6) != 1).any():
    raise ValueError("Some correlation with itself is not 1.")
print("Known case OK: every correlation with itself is 1")

A second check, from 3a: Apple's volatility in your table must be the same as the number you saw in 3a, cell 5 (28.1%). Run the cell and compare.

In [ ]:
apple_vol = risk_table.loc["AAPL", "Volatility"]
print(f"Apple volatility in the table: {apple_vol:.1%}")

## Step 6 - Send it to the MD

`.to_excel("risk_table.xlsx")` writes the table to Excel. In Colab, the second part downloads it.

In [ ]:
risk_table.to_excel("risk_table.xlsx")

try:
    from google.colab import files
    files.download("risk_table.xlsx")
except ImportError:
    print("Not in Colab: the file risk_table.xlsx is in the current folder.")

## What you learned

**Python**
- **DataFrame from columns**: `pd.DataFrame({"Volatility": vol, ...})`, one column per measure, one row per asset, built from a **dictionary**.
- **Sort**: `.sort_values("Sharpe", ascending=False)`, best first. **Round**: `.round(3)`, readable (0.196 is 19.6%), the original table is unchanged.
- **Statistic per column** again: `returns.std()`, `returns.cov()`, `.var()`. **Row by position**: `prices.iloc[-1]`, `prices.iloc[0]`.
- **Guard**: `if ...: raise ValueError(...)` stops the code when the known case fails.
- **Export**: `.to_excel("risk_table.xlsx")`.

**Finance**
- Annual return = compound growth: `(last / first) ** (252 / days) - 1`.
- Beta = covariance with the index / variance of the index. Sharpe = `(return - rf) / volatility`.
- Known case: the beta of the S&P 500 is 1.00. A check you can always run.
- Only Apple beats the index once risk is counted: Sharpe 0.59 against 0.43.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right